# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd().parent / "database" / "flasheats.db").exists():
    BASE = Path.cwd().parent   # solution notebooks live in a sub-folder of the pack

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: C:\Users\harsh\Documents\SST\Forward Deployed Engineer\flasheats-classroom-pack


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders=orders.drop_duplicates("order_id",keep="first")
late_order=outcomes[outcomes.late_flag==1].order_id.iloc[0]
on_time_order=outcomes[outcomes.late_flag==0].order_id.iloc[0]
intervention_order=interventions.order_id.iloc[0]
print("late",late_order,"on-time",on_time_order,"intervention",intervention_order)

# Solution: see the cells below

late O00001 on-time O00003 intervention O00781


In [4]:
# Challenge 1 - rebuild an order's lifecycle from every source
with open(BASE / "data" / "driver_events.json") as f:
    driver_ev = pd.DataFrame([{"driver_id": d["driver_id"], **e} for d in json.load(f) for e in d["events"]])
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv").drop_duplicates()
with open(BASE / "api" / "dispatch_data.json") as f:
    dispatch = pd.DataFrame(json.load(f))

def build_order_timeline(order_id):
    o = base_orders[base_orders.order_id == order_id].iloc[0]
    rows = [
        (o.created_at, "ORDER_CREATED", f"customer {o.customer_id}", "orders (SQLite)"),
        (o.promised_eta, "ETA_PROMISED (target, not an event)", "platform", "orders (SQLite)"),
        (o.pickup_at, "PICKED_UP", f"driver {o.driver_id}", "orders (SQLite)"),
        (o.actual_delivery_at, "DELIVERED", f"driver {o.driver_id}", "orders (SQLite)"),
    ]
    d = dispatch[dispatch.order_id == order_id].iloc[0]
    rows.append((d.estimated_pickup_at, "ESTIMATED_PICKUP (target)", "dispatch", "dispatch API"))
    if pd.notna(d.reassigned_at):
        rows.append((d.reassigned_at, f"DRIVER_REASSIGNED {d.original_driver_id} -> {d.driver_id}", "dispatch", "dispatch API"))

    for e in driver_ev[driver_ev.order_id == order_id].itertuples():
        rows.append((e.timestamp, e.type.upper(), f"driver {e.driver_id}", "driver_events.json"))
    for s in restaurant_status[restaurant_status.order_id == order_id].itertuples():
        rows.append((s.last_updated_at, f"RESTAURANT_STATUS = {s.status}", f"restaurant {s.restaurant_id}", "restaurant_status.csv"))
    for a in customer_actions[customer_actions.order_id == order_id].itertuples():
        rows.append((a.action_at, a.action_type, f"customer {a.customer_id}", "customer_app_actions.csv"))
    for t in tickets[tickets.order_id == order_id].drop_duplicates("ticket_id").itertuples():
        rows.append((t.created_at, f"SUPPORT_TICKET ({t.category.strip()})", "customer", "support_tickets.csv"))
    for i in interventions[interventions.order_id == order_id].itertuples():
        rows.append((i.intervention_at, f"INTERVENTION {i.intervention_type} ({i.reason})", i.initiated_by, "order_interventions.csv"))

    tl = pd.DataFrame(rows, columns=["event_time", "event_type", "actor", "source_system"])
    tl["event_time"] = pd.to_datetime(tl["event_time"], format="mixed", errors="coerce")
    tl = tl.dropna(subset=["event_time"]).sort_values("event_time").reset_index(drop=True)
    tl.insert(1, "min_from_order", ((tl.event_time - pd.to_datetime(o.created_at)).dt.total_seconds() / 60).round(1))
    return tl

for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("INTERVENTION", intervention_order)]:
    out = outcomes[outcomes.order_id == oid].iloc[0]
    print(f"\n{label}: {oid} -> {out.outcome_bucket}, delay {out.delay_min} min")
    display(build_order_timeline(oid))


ON TIME: O00003 -> delivered_on_time, delay -4.41 min


,event_time,min_from_order,event_type,actor,source_system
0,2026-08-01 19:35:00.000000,0.0,ORDER_CREATED,customer C0855,orders (SQLite)
1,2026-08-01 19:35:00.002443,0.0,ASSIGNED,driver D039,driver_events.json
2,2026-08-01 19:44:37.017130,9.6,GPS_PING,driver D039,driver_events.json
3,2026-08-01 19:48:00.000000,13.0,RESTAURANT_STATUS = preparing,restaurant R010,restaurant_status.csv
4,2026-08-01 19:53:00.000000,18.0,ESTIMATED_PICKUP (target),dispatch,dispatch API
5,2026-08-01 19:54:14.031817,19.2,GPS_PING,driver D039,driver_events.json
6,2026-08-01 19:57:50.820366,22.8,PICKED_UP,driver D039,orders (SQLite)
7,2026-08-01 19:57:50.820366,22.8,PICKED_UP,driver D039,driver_events.json
8,2026-08-01 20:03:51.046504,28.9,GPS_PING,driver D039,driver_events.json
9,2026-08-01 20:13:28.061191,38.5,DELIVERED,driver D039,orders (SQLite)



LATE: O00001 -> delivered_late, delay 10.82 min


,event_time,min_from_order,event_type,actor,source_system
0,2026-08-13 12:56:00.000000,0.0,ORDER_CREATED,customer C0168,orders (SQLite)
1,2026-08-13 13:01:09.659644,5.2,ASSIGNED,driver D103,driver_events.json
2,2026-08-13 13:08:00.000000,12.0,INTERVENTION DRIVER_REASSIGNMENT (driver_unavailable),dispatch,order_interventions.csv
3,2026-08-13 13:14:00.000000,18.0,ESTIMATED_PICKUP (target),dispatch,dispatch API
4,2026-08-13 13:14:42.222353,18.7,GPS_PING,driver D103,driver_events.json
5,2026-08-13 13:21:00.000000,25.0,ETA_VIEWED,customer C0168,customer_app_actions.csv
6,2026-08-13 13:28:14.785062,32.2,GPS_PING,driver D103,driver_events.json
7,2026-08-13 13:35:49.825561,39.8,PICKED_UP,driver D103,orders (SQLite)
8,2026-08-13 13:35:49.825561,39.8,PICKED_UP,driver D103,driver_events.json
9,2026-08-13 13:41:47.347771,45.8,GPS_PING,driver D103,driver_events.json



INTERVENTION: O00781 -> delivered_late, delay 13.07 min


,event_time,min_from_order,event_type,actor,source_system
0,2026-08-22 12:33:00.000000,0.0,ORDER_CREATED,customer C0103,orders (SQLite)
1,2026-08-22 12:40:42.931820,7.7,ASSIGNED,driver D103,driver_events.json
2,2026-08-22 12:51:00.000000,18.0,ESTIMATED_PICKUP (target),dispatch,dispatch API
3,2026-08-22 12:54:00.000000,21.0,ETA_VIEWED,customer C0103,customer_app_actions.csv
4,2026-08-22 12:57:00.000000,24.0,INTERVENTION PRIORITY_DISPATCH (late_risk),operations,order_interventions.csv
5,2026-08-22 13:04:00.000000,31.0,RESTAURANT_STATUS = ready,restaurant R019,restaurant_status.csv
6,2026-08-22 13:07:42.051142,34.7,GPS_PING,driver D103,driver_events.json
7,2026-08-22 13:11:17.309762,38.3,PICKED_UP,driver D103,orders (SQLite)
8,2026-08-22 13:11:17.309762,38.3,PICKED_UP,driver D103,driver_events.json
9,2026-08-22 13:22:41.170463,49.7,GPS_PING,driver D103,driver_events.json


### Challenge 1 — Answer

Each timeline combines **7 sources**: orders, dispatch API, driver events, restaurant status, app actions, support tickets and interventions. Reading the three orders side by side shows how the workflow breaks down:

- **On time (O00003):**
  - Assigned almost immediately and picked up ~23 min after ordering, close to dispatch's estimated pickup.
  - Delivered ~4 min before the promise.
  - The restaurant status says `preparing`, stamped *before* pickup and never updated afterwards. Even on a good order the status feed is stale.
- **Late (O00001):**
  - Pickup happened ~40 min after the order, **~22 min after dispatch's estimated pickup**. The order was already behind before the driver left the restaurant.
  - The ride itself was normal (~47 min). Delivered 10.8 min late.
  - The customer checked the ETA, then **opened support**.
  - A `DRIVER_REASSIGNMENT` intervention is logged, **but dispatch shows no reassignment and the same driver throughout**. The intervention log and dispatch contradict each other.
- **Intervention (O00781):**
  - Pickup was due at 18 min. At 24 min, ops logged a `PRIORITY_DISPATCH` for `late_risk`, so the risk was spotted early.
  - The restaurant marked the food `ready` at 31 min, but pickup only happened at 38 min, ~20 min behind estimate. Delivered 13 min late.
  - The intervention came after the pickup estimate had already been missed, and pickup still slipped another ~14 min.
  - Its restaurant (R019) also has invalid coordinates.

`PICKED_UP` and `DELIVERED` appear twice in each timeline because both the orders table and the driver app record them. For these orders the two agree exactly.

**Pattern:**
- Delay builds up **between order and pickup**. That is exactly the stage with **no observed events** (no driver-arrived, no food-ready).
- Customer interactions (ETA views, support) and interventions sit around that gap.

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [5]:
sources={"orders":orders,"customer_actions":customer_actions,"support_tickets":tickets,"interventions":interventions,"outcomes":outcomes}
for name,df in sources.items():
    print(name,df.shape); display(df.head(2))
# Solution: see the cells below

orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


In [6]:
# Challenge 2 - check the keys, grain and relationships of the model
model = [
    # table,            dataframe,        primary key,       foreign keys,                    grain
    ("customers",        customers,        "customer_id",     [],                              "one customer"),
    ("orders",           base_orders,      "order_id",        ["customer_id", "restaurant_id", "driver_id"], "one order"),
    ("customer_actions", customer_actions, "action_id",       ["order_id", "customer_id"],     "one in-app action on an order"),
    ("support_tickets",  tickets,          "ticket_id",       ["order_id"],                    "one support ticket"),
    ("interventions",    interventions,    "intervention_id", ["order_id"],                    "one operational action on an order"),
    ("outcomes",         outcomes,         "order_id",        ["order_id"],                    "one final result per order"),
]
display(pd.DataFrame([{"table": n, "grain": g, "rows": len(df), "primary_key": pk,
                       "pk_unique": df[pk].is_unique, "foreign_keys": ", ".join(fk) or "-"}
                      for n, df, pk, fk, g in model]))

# Foreign-key coverage: does every child row point to a real parent?
fk_checks = [
    ("orders.customer_id -> customers", base_orders.customer_id, customers.customer_id),
    ("customer_actions.order_id -> orders", customer_actions.order_id, base_orders.order_id),
    ("support_tickets.order_id -> orders", tickets.order_id, base_orders.order_id),
    ("interventions.order_id -> orders", interventions.order_id, base_orders.order_id),
    ("outcomes.order_id -> orders", outcomes.order_id, base_orders.order_id),
]
for name, child, parent in fk_checks:
    print(f"{name:40} {child.isin(parent).mean():6.1%} mapped | {child.isna().sum()} null")

# Does the action's customer match the order's customer?
chk = customer_actions.merge(base_orders[["order_id", "customer_id"]], on="order_id", suffixes=("", "_order"))
print("customer_actions with a different customer than the order:", (chk.customer_id != chk.customer_id_order).sum())

# Cardinalities of the one-to-many relationships
print("\nOrders per customer (max):", base_orders.customer_id.value_counts().max())
print("Actions per order (max):", customer_actions.order_id.value_counts().max())
print("Interventions per order (max):", interventions.order_id.value_counts().max())
print("Tickets per order (max):", tickets.order_id.value_counts().max())

,table,grain,rows,primary_key,pk_unique,foreign_keys
0,customers,one customer,900,customer_id,True,-
1,orders,one order,1600,order_id,True,"customer_id, restaurant_id, driver_id"
2,customer_actions,one in-app action on an order,2365,action_id,True,"order_id, customer_id"
3,support_tickets,one support ticket,202,ticket_id,False,order_id
4,interventions,one operational action on an order,430,intervention_id,True,order_id
5,outcomes,one final result per order,1600,order_id,True,order_id


orders.customer_id -> customers          100.0% mapped | 0 null
customer_actions.order_id -> orders      100.0% mapped | 0 null


support_tickets.order_id -> orders        98.5% mapped | 3 null
interventions.order_id -> orders         100.0% mapped | 0 null
outcomes.order_id -> orders              100.0% mapped | 0 null
customer_actions with a different customer than the order: 0

Orders per customer (max): 8
Actions per order (max): 5
Interventions per order (max): 1


Tickets per order (max): 2


In [7]:
# One canonical "customer interaction" entity instead of two source tables
interactions = pd.concat([
    customer_actions.rename(columns={"action_id": "interaction_id", "action_type": "interaction_type", "action_at": "interaction_at"})
        [["interaction_id", "order_id", "customer_id", "interaction_type", "interaction_at", "channel"]],
    tickets.drop_duplicates("ticket_id").dropna(subset=["order_id"])
        .merge(base_orders[["order_id", "customer_id"]], on="order_id")
        .assign(interaction_type="SUPPORT_TICKET", channel="support")
        .rename(columns={"ticket_id": "interaction_id", "created_at": "interaction_at"})
        [["interaction_id", "order_id", "customer_id", "interaction_type", "interaction_at", "channel"]],
], ignore_index=True)

print("interactions:", interactions.shape, "| interaction_id unique:", interactions.interaction_id.is_unique)
display(interactions.groupby(["channel", "interaction_type"]).size().to_frame("rows"))

# Two support channels that should describe the same thing - do they overlap?
app_support = set(customer_actions.loc[customer_actions.action_type == "SUPPORT_OPENED", "order_id"])
ticket_orders = set(tickets.order_id.dropna())
print("Orders with in-app SUPPORT_OPENED:", len(app_support), "| with a ticket:", len(ticket_orders), "| both:", len(app_support & ticket_orders))

interactions: (2563, 6) | interaction_id unique: True


rows
channel    interaction_type      
mobile_app CANCEL_ATTEMPTED    10
           ETA_VIEWED        2217
           SUPPORT_OPENED     138
support    SUPPORT_TICKET     198

Orders with in-app SUPPORT_OPENED: 138 | with a ticket: 198 | both: 43


### Challenge 2 — Answer

**Canonical project model**, organised around the order workflow rather than the source systems:

| Table (entity) | Primary key | Important foreign keys | Grain |
|---|---|---|---|
| `customer` | `customer_id` | — | one customer (900) |
| `order` | `order_id` (deduped, 1,600) | `customer_id`, `restaurant_id`, `driver_id` (should be the **current** driver from dispatch) | one order |
| `customer_interaction` | `interaction_id` | `order_id`, `customer_id` | one customer touchpoint on an order: app action (`ETA_VIEWED`, `SUPPORT_OPENED`, `CANCEL_ATTEMPTED`) **or** support ticket |
| `order_intervention` | `intervention_id` | `order_id` | one operational action (`DRIVER_REASSIGNMENT`, `PRIORITY_DISPATCH`, `RESTAURANT_CONTACT`, `CUSTOMER_CREDIT`) |
| `order_outcome` | `order_id` (1:1 with order) | `order_id` | one final result per order (`final_status`, `late_flag`, `delay_min`) |
| *reference:* `restaurant`, `driver` | `restaurant_id`, `driver_id` | — | one restaurant / driver |

**Relationships:**
- customer 1 → N order (up to 8 orders per customer)
- order 1 → N customer_interaction (up to 5 actions)
- order 1 → N order_intervention (currently at most 1 per order)
- order 1 → 1 order_outcome

**Checks:**
- Every foreign key maps 100%.
- The action's customer always matches the order's customer.
- The two problems found are both **source issues, not model issues**:
  - `ticket_id` is not unique (1 duplicate).
  - 3 tickets have no `order_id`.

**Why this is better than mirroring every source table:**
1. **It answers the business question directly.** The project asks "interaction → intervention → outcome". Each of those is one entity keyed on `order_id`, so the question becomes a simple join instead of re-reconciling 7 sources every time.
2. **One concept, one table.**
   - In-app support and tickets are both *customer interactions*, so they become one table with a `channel` column.
   - Mirrored, they stay two tables that barely overlap: only 43 orders appear in both, out of 138 in-app and 198 ticket orders. Every analyst would then define "support contact" differently.
3. **The grain is explicit and enforced.** Aggregating 1:N tables to order level before joining avoids fan-out double counting, and the duplicate orders and tickets are resolved once.
4. **It is small and stable.** New sources (e.g. a future `driver_arrived` event) plug in as new event types, not new structures.
5. **Source-specific noise stays in staging.** GPS pings, raw API pages and the category spellings stay out of the model.

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [8]:
actions_by_order=(customer_actions.groupby("order_id").agg(action_count=("action_id","count")).reset_index())
# Solution: see the cells below

In [9]:
# Challenge 3 - one row per order: interaction -> intervention -> outcome
# Aggregate every one-to-many table to order level BEFORE joining (avoids fan-out)
action_counts = (customer_actions.pivot_table(index="order_id", columns="action_type", values="action_id",
                                              aggfunc="count", fill_value=0).reset_index())
ticket_counts = (tickets.drop_duplicates("ticket_id").dropna(subset=["order_id"])
                 .groupby("order_id").size().rename("ticket_count").reset_index())
iv_by_order = (interventions.groupby("order_id")
               .agg(intervention_count=("intervention_id", "count"),
                    intervention_types=("intervention_type", lambda s: ", ".join(sorted(s.unique()))))
               .reset_index())

order_model = (base_orders[["order_id", "customer_id", "restaurant_id"]]
               .merge(outcomes, on="order_id", how="left")
               .merge(action_counts, on="order_id", how="left")
               .merge(ticket_counts, on="order_id", how="left")
               .merge(iv_by_order, on="order_id", how="left"))

count_cols = ["ETA_VIEWED", "SUPPORT_OPENED", "CANCEL_ATTEMPTED", "ticket_count", "intervention_count"]
order_model[count_cols] = order_model[count_cols].fillna(0).astype(int)
order_model["intervention_types"] = order_model["intervention_types"].fillna("none")

# support_opened = in-app support OR a support ticket (either channel counts)
order_model["support_opened"] = ((order_model.SUPPORT_OPENED > 0) | (order_model.ticket_count > 0)).astype(int)
order_model["cancel_attempted"] = (order_model.CANCEL_ATTEMPTED > 0).astype(int)
order_model = order_model.rename(columns={"final_status_norm": "final_status"})

# Grain check: still exactly one row per order
assert order_model.order_id.is_unique and len(order_model) == len(base_orders)

cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count",
        "intervention_types", "final_status", "late_flag", "delay_min"]
display(order_model[cols].head(10))

,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min
0,O00001,C0168,1,0,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82
1,O00002,C0043,0,0,0,none,delivered,1.0,3.97
2,O00003,C0855,0,0,0,none,delivered,0.0,-4.41
3,O00004,C0229,0,0,0,none,cancelled,NaN,NaN
4,O00005,C0040,0,0,0,none,delivered,1.0,21.02
5,O00006,C0152,0,0,0,none,delivered,1.0,7.30
6,O00007,C0875,0,0,0,none,delivered,1.0,4.27
7,O00008,C0223,1,0,1,CUSTOMER_CREDIT,delivered,1.0,26.86
8,O00009,C0685,0,0,0,none,delivered,1.0,0.69
9,O00010,C0530,0,0,0,none,delivered,0.0,-5.79


In [10]:
# Answer the four questions from the order-level table
late = order_model[order_model.late_flag == 1]
print(f"1. Late orders with a support interaction: {late.support_opened.sum()} of {len(late)} ({late.support_opened.mean():.1%})")
print(f"   in-app SUPPORT_OPENED: {(late.SUPPORT_OPENED > 0).sum()} | ticket: {(late.ticket_count > 0).sum()}")

has_iv = order_model.intervention_count > 0
print(f"\n2. Orders with an intervention: {has_iv.sum()} of {len(order_model)} ({has_iv.mean():.1%}), max per order = {order_model.intervention_count.max()}")

print("\n3. Most common intervention:")
display(interventions.intervention_type.value_counts().to_frame("count"))

# 4. Frustrated = customer asked for help or tried to cancel
frustrated = order_model[(order_model.support_opened == 1) | (order_model.cancel_attempted == 1)]
no_help = frustrated[frustrated.intervention_count == 0]
print(f"4. Frustrated journeys: {len(frustrated)} | with NO intervention: {len(no_help)}")
display(no_help.outcome_bucket.value_counts().to_frame("orders"))
print("Worst frustrated journeys with no intervention:")
display(no_help.sort_values("delay_min", ascending=False)[cols + ["ETA_VIEWED", "ticket_count"]].head(10))

1. Late orders with a support interaction: 258 of 843 (30.6%)
   in-app SUPPORT_OPENED: 138 | ticket: 163

2. Orders with an intervention: 430 of 1600 (26.9%), max per order = 1

3. Most common intervention:


,count
intervention_type,
DRIVER_REASSIGNMENT,155
RESTAURANT_CONTACT,116
PRIORITY_DISPATCH,95
CUSTOMER_CREDIT,64


4. Frustrated journeys: 294 | with NO intervention: 219


,orders
outcome_bucket,
delivered_late,192
delivered_on_time,26
unknown,1


Worst frustrated journeys with no intervention:


,order_id,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types,final_status,late_flag,delay_min,ETA_VIEWED,ticket_count
193,O00194,C0771,1,0,0,none,delivered,1.0,38.34,0,1
736,O00737,C0824,1,0,0,none,delivered,1.0,36.28,2,0
302,O00303,C0411,1,0,0,none,delivered,1.0,33.16,3,1
210,O00211,C0629,1,0,0,none,delivered,1.0,32.38,3,1
1234,O01235,C0772,1,0,0,none,delivered,1.0,32.00,0,1
976,O00977,C0254,1,0,0,none,delivered,1.0,31.92,1,1
994,O00995,C0746,1,0,0,none,delivered,1.0,31.72,1,1
1196,O01197,C0032,1,0,0,none,delivered,1.0,31.61,0,1
892,O00893,C0556,1,0,0,none,delivered,1.0,30.32,3,0
1143,O01144,C0788,1,0,0,none,delivered,1.0,30.13,3,1


### Challenge 3 — Answer

The order-level table has **1,600 rows, one per order** (checked with an assert). All 1:N sources were aggregated before the join.

1. **Late orders with a support interaction: 258 of 843 (30.6%).**
   - 138 came through in-app `SUPPORT_OPENED` and 163 through tickets, with little overlap between the two.
   - Every in-app `SUPPORT_OPENED` order was late (138/138).
   - ~70% of late orders were never raised with support, so support volume badly undercounts the problem.
2. **Orders with an intervention: 430 of 1,600 (26.9%).** Each has exactly one intervention.
3. **Most common intervention: `DRIVER_REASSIGNMENT` (155)**, then `RESTAURANT_CONTACT` (116), `PRIORITY_DISPATCH` (95) and `CUSTOMER_CREDIT` (64).
4. **Frustrated journeys with no intervention: 219 of 294.** "Frustrated" means support opened by either channel, or a cancel attempt.
   - **192 of them were delivered late** (26 on time, 1 unknown).
   - The worst were 30–38 min late.
   - 7 of the 10 customers who tried to cancel got no intervention.
   - This is the clearest operational gap: **the customer signal exists, but it does not trigger an intervention**.

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [11]:
# Example starting point:
# valid_outcomes=outcomes[outcomes.late_flag.notna()]
# late_delivery_rate=valid_outcomes.late_flag.mean()

# Solution: see the cells below

In [12]:
# Challenge 4 - calculate the selected metrics
om = order_model.merge(base_orders[["order_id", "created_at", "pickup_at"]], on="order_id")
om["first_mile_min"] = (pd.to_datetime(om.pickup_at, format="mixed")
                        - pd.to_datetime(om.created_at, format="mixed")).dt.total_seconds() / 60

known = om[om.late_flag.notna()]        # delivered orders with a delivery timestamp

# Intervention timing: operational actions (not credits) made before pickup
iv_t = interventions.merge(base_orders[["order_id", "pickup_at"]], on="order_id")
iv_t["before_pickup"] = pd.to_datetime(iv_t.intervention_at, format="mixed") <= pd.to_datetime(iv_t.pickup_at, format="mixed")
operational = iv_t[iv_t.intervention_type != "CUSTOMER_CREDIT"]

metrics = pd.DataFrame([
    ("Late Delivery Rate", "Outcome (project KPI)", known.late_flag.mean()),
    ("Severe Late Rate (> 10 min)", "Outcome", (known.delay_min > 10).mean()),
    ("Median first-mile time (min, order -> pickup)", "Workflow driver", om.first_mile_min.median()),
    ("Support Contact Rate", "Customer interaction", om.support_opened.mean()),
    ("Proactive Intervention Rate (before pickup)", "Intervention", operational.before_pickup.mean()),
], columns=["metric", "type", "value"])
metrics["value"] = metrics["value"].round(3)
display(metrics)

# Why first-mile time earns its place: it moves with the KPI
known = known.assign(first_mile_band=pd.cut(known.first_mile_min, [0, 20, 25, 30, 35, 200],
                                            labels=["<20", "20-25", "25-30", "30-35", "35+"]))
display(known.groupby("first_mile_band", observed=True).agg(orders=("order_id", "count"),
                                                             late_rate=("late_flag", "mean")).round(2))

,metric,type,value
0,Late Delivery Rate,Outcome (project KPI),0.564
1,Severe Late Rate (> 10 min),Outcome,0.233
2,"Median first-mile time (min, order -> pickup)",Workflow driver,26.140
3,Support Contact Rate,Customer interaction,0.183
4,Proactive Intervention Rate (before pickup),Intervention,0.689


,orders,late_rate
first_mile_band,,
<20,345,0.02
20-25,327,0.31
25-30,305,0.74
30-35,204,0.97
35+,314,0.99


### Challenge 4 — Answer

| # | Metric | Formula | Grain | Why it matters | Relationship to the KPI | Value |
|---|---|---|---|---|---|---|
| 1 | **Late Delivery Rate** *(outcome)* | orders with `delay_min > 0` ÷ delivered orders with a known delay | order | This is the project KPI and the customer promise | **It *is* the KPI** | **56.4%** |
| 2 | **Severe Late Rate** *(outcome)* | orders with `delay_min > 10` ÷ delivered orders with a known delay | order | Separates "1 min late" from a real failure. This is the level where customers complain. | A guard-rail: the KPI could improve while severe cases stay the same | **23.3%** |
| 3 | **Median first-mile time** *(workflow driver)* | median(`pickup_at − created_at`) | order | This is where delay builds up. The late rate rises from ~2% (<20 min) to ~99% (35+ min). | A **leading indicator**: cutting first-mile time is the main lever on the KPI | **~26 min** |
| 4 | **Support Contact Rate** *(customer interaction)* | orders with in-app support or a ticket ÷ all orders | order | Measures the customer pain and the cost of lateness | A lagging symptom: 88% of contacted orders were late | **18.3%** |
| 5 | **Proactive Intervention Rate** *(intervention)* | operational interventions made **before pickup** ÷ all operational interventions (excludes credits) | intervention | Only interventions made *before* the delay locks in can change the outcome | The controllable lever. Credits after delivery do nothing for the KPI. | **~69%** |

**Notes:**
- Metrics 1–2 are the **outcome**, 3 is the **driver**, 4 is the **customer reaction** and 5 is what **operations controls**.
- This is deliberately a small set, and each metric has one formula and one grain.

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [13]:
# Solution: see the cells below
# Reminder: association != causation

In [14]:
# Challenge 5 - joins and aggregations on the order-level model
import numpy as np

known = order_model[order_model.late_flag.notna()].copy()
known["has_intervention"] = known.intervention_count > 0

def rates(df, by):
    return (df.groupby(by).agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
                               median_delay=("delay_min", "median")).round(3))

# A. Do orders with support interactions have higher delay?
print("A. Support interaction vs delay")
display(rates(known, "support_opened"))
so = customer_actions[customer_actions.action_type == "SUPPORT_OPENED"].merge(base_orders[["order_id", "promised_eta"]], on="order_id")
so_vs_eta = (pd.to_datetime(so.action_at, format="mixed") - pd.to_datetime(so.promised_eta, format="mixed")).dt.total_seconds() / 60
print(f"   In-app support opened a median {so_vs_eta.median():.1f} min AFTER the promised ETA")

# B. Late rate with vs without intervention
print("\nB. With vs without intervention")
display(rates(known, "has_intervention"))

A. Support interaction vs delay


,orders,late_rate,median_delay
support_opened,,,
0,1203,0.486,-0.250
1,292,0.884,12.285


   In-app support opened a median 11.0 min AFTER the promised ETA

B. With vs without intervention


,orders,late_rate,median_delay
has_intervention,,,
False,1091,0.564,1.39
True,404,0.564,1.54


In [15]:
# C. Which intervention type has the lowest late rate - and when does it happen?
iv = interventions.merge(known[["order_id", "late_flag", "delay_min"]], on="order_id")
iv = iv.merge(base_orders[["order_id", "pickup_at", "actual_delivery_at"]], on="order_id")
at = pd.to_datetime(iv.intervention_at, format="mixed")
iv["timing"] = np.select([at <= pd.to_datetime(iv.pickup_at, format="mixed"),
                          at <= pd.to_datetime(iv.actual_delivery_at, format="mixed")],
                         ["before pickup", "in transit"], "after delivery")

print("C. Late rate by intervention type")
display(iv.groupby("intervention_type").agg(orders=("order_id", "count"), late_rate=("late_flag", "mean"),
                                            median_delay=("delay_min", "median"),
                                            share_before_pickup=("timing", lambda s: (s == "before pickup").mean()))
          .round(3).sort_values("late_rate"))
display(pd.crosstab(iv.intervention_type, iv.timing))

# Only interventions made before pickup, split by reason
print("Before-pickup interventions by reason:")
display(iv[iv.timing == "before pickup"].groupby(["intervention_type", "reason"])
          .agg(orders=("order_id", "count"), late_rate=("late_flag", "mean")).round(2))

C. Late rate by intervention type


,orders,late_rate,median_delay,share_before_pickup
intervention_type,,,,
PRIORITY_DISPATCH,86,0.512,0.33,0.663
RESTAURANT_CONTACT,108,0.528,0.80,0.602
DRIVER_REASSIGNMENT,147,0.578,1.60,0.748
CUSTOMER_CREDIT,63,0.667,2.93,0.016


timing,after delivery,before pickup,in transit
intervention_type,,,
CUSTOMER_CREDIT,45,1,17
DRIVER_REASSIGNMENT,0,110,37
PRIORITY_DISPATCH,0,57,29
RESTAURANT_CONTACT,0,65,43


Before-pickup interventions by reason:


orders  late_rate
intervention_type   reason                                
CUSTOMER_CREDIT     support_resolution        1       1.00
DRIVER_REASSIGNMENT capacity_rebalance       32       0.75
                    driver_unavailable       37       0.73
                    slow_progress            41       0.73
PRIORITY_DISPATCH   late_risk                23       0.61
                    manual_ops_review        16       0.75
                    vip_customer             18       0.78
RESTAURANT_CONTACT  customer_escalation      21       0.71
                    prep_delay               23       0.65
                    status_stale             21       0.71

In [16]:
# D. Which restaurants contribute the most late orders?
rest = (known.groupby("restaurant_id")
             .agg(orders=("order_id", "count"), late_orders=("late_flag", "sum"), late_rate=("late_flag", "mean"))
             .sort_values(["late_orders", "late_rate"], ascending=False))
rest["share_of_all_late"] = rest.late_orders / rest.late_orders.sum()
print("D. Top 10 restaurants by late orders")
display(rest.head(10).round(3))
top10 = rest.head(10)
print(f"   Top 10 of {len(rest)} restaurants: {top10.late_orders.sum() / rest.late_orders.sum():.1%} of late orders, "
      f"{top10.orders.sum() / rest.orders.sum():.1%} of orders")

# E. Support interaction + intervention + still late
e = known[(known.support_opened == 1) & known.has_intervention & (known.late_flag == 1)].copy()
print(f"\nE. Support + intervention + still late: {len(e)} orders")
display(e.intervention_types.value_counts().to_frame("orders"))

# Did the intervention come before or after the customer asked for help?
first_support = pd.concat([
    customer_actions[customer_actions.action_type == "SUPPORT_OPENED"][["order_id", "action_at"]].rename(columns={"action_at": "t"}),
    tickets.dropna(subset=["order_id"])[["order_id", "created_at"]].rename(columns={"created_at": "t"}),
]).assign(t=lambda x: pd.to_datetime(x.t, format="mixed")).groupby("order_id").t.min()
e_iv = interventions[interventions.order_id.isin(e.order_id)].set_index("order_id")
e_iv["after_support"] = pd.to_datetime(e_iv.intervention_at, format="mixed") > first_support.reindex(e_iv.index)
print(f"   Intervention came AFTER the first support contact in {e_iv.after_support.mean():.0%} of these journeys")
display(e.sort_values("delay_min", ascending=False)[["order_id", "restaurant_id", "intervention_types", "delay_min"]].head(8))

D. Top 10 restaurants by late orders


,orders,late_orders,late_rate,share_of_all_late
restaurant_id,,,,
R050,35,22.0,0.629,0.026
R024,25,20.0,0.800,0.024
R051,29,20.0,0.690,0.024
R004,30,20.0,0.667,0.024
R023,32,20.0,0.625,0.024
R030,38,20.0,0.526,0.024
R045,29,19.0,0.655,0.023
R011,31,19.0,0.613,0.023
R018,27,18.0,0.667,0.021


   Top 10 of 60 restaurants: 23.3% of late orders, 20.4% of orders

E. Support + intervention + still late: 67 orders


,orders
intervention_types,
DRIVER_REASSIGNMENT,26
CUSTOMER_CREDIT,15
PRIORITY_DISPATCH,14
RESTAURANT_CONTACT,12


   Intervention came AFTER the first support contact in 25% of these journeys


,order_id,restaurant_id,intervention_types,delay_min
100,O00101,R007,CUSTOMER_CREDIT,85.51
197,O00198,R016,RESTAURANT_CONTACT,35.23
496,O00497,R032,PRIORITY_DISPATCH,31.78
612,O00613,R023,PRIORITY_DISPATCH,27.18
7,O00008,R010,CUSTOMER_CREDIT,26.86
301,O00302,R019,PRIORITY_DISPATCH,26.32
543,O00544,R023,DRIVER_REASSIGNMENT,24.92
861,O00862,R025,DRIVER_REASSIGNMENT,24.10


### Challenge 5 — Answer

**A. Do orders with support interactions have higher delay? Yes, much higher.**
- The late rate is **88%** with support vs **49%** without. Median delay is **+12.3 min** vs **−0.3 min**.
- In-app support is opened a median ~11 min *after* the promised ETA has passed.
- *What it tells the business:* support contact is a **lagging, reactive** signal. By the time a customer asks, the order is already late.
- *What it does NOT prove:* that support causes delay (the causation runs the other way), or that support is a good early-warning trigger. It comes too late to prevent the lateness.

**B. Late rate with vs without intervention: 56.4% vs 56.4%. No difference.**
- *What it tells the business:* today's interventions are not visibly reducing lateness overall.
- *What it does NOT prove:* that interventions don't work. Interventions are **targeted at at-risk orders** (selection bias), so an intervened order that lands at "average" may have been saved from something worse. We have no counterfactual and no risk score at the time of intervention.

**C. Lowest late rate by intervention type: `PRIORITY_DISPATCH` (~51%)**, then `RESTAURANT_CONTACT` (~53%), `DRIVER_REASSIGNMENT` (~58%) and `CUSTOMER_CREDIT` (~67%).
- Most `CUSTOMER_CREDIT`s happen **after delivery** (45 of 63). They are service recovery and cannot change lateness, so they should be excluded from any "what reduces lateness" comparison.
- Interventions made **before pickup** are late **61–78%** of the time, which is *above* the 56% baseline. This fits interventions being aimed at orders that were already at risk.
- *What it tells the business:* priority dispatch and restaurant contact, done **before pickup**, are the most promising levers to test.
- *What it does NOT prove:* that priority dispatch *causes* better outcomes. The groups are small (86–147 orders with a known outcome), are triggered for different reasons (`vip_customer` vs `late_risk`), and were not randomised. A controlled A/B test is needed.

**D. Restaurants with the most late orders:**
- **R050 (22), then R004, R024, R023, R030 and R051 (20 each).**
- The top 10 restaurants account for ~23% of late orders but also ~20% of volume. Lateness is spread widely, not concentrated in a few bad restaurants.
- Some have a *high rate* (R024 80%, R051 69%). Others are mostly *high volume* (R030 53%, below average).
- *What it tells the business:* rank by **both** count and rate. R024 and R004 are the most urgent partner conversations.
- *What it does NOT prove:* that the restaurant is at fault. First-mile delay could be the driver arriving late, and we have no food-ready or driver-arrived event to tell the two apart.

**E. Support + intervention + still late: 67 journeys.**
- In 75% of these journeys the intervention came *before* the customer's support contact, so it was proactive but did not prevent the delay.
- The worst one (O00101, 85 min) is one of the impossible-ETA orders, so its "delay" is a data artifact.
- *What it tells the business:* these are the most expensive failures. The customer is upset, ops spent effort, and the order was still late. They are the best case studies for "why did the intervention not work?"
- *What it does NOT prove:* that the intervention made things worse. These orders were probably the most at-risk to begin with.

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

In [17]:
# Challenge 6 - evidence behind the KPI tree and the key limitation
reassign_api = set(dispatch.loc[dispatch.reassigned_at.notna(), "order_id"])
reassign_iv = set(interventions.loc[interventions.intervention_type == "DRIVER_REASSIGNMENT", "order_id"])
print("Driver reassignments - dispatch API:", len(reassign_api), "| intervention log:", len(reassign_iv),
      "| in both:", len(reassign_api & reassign_iv))

# Which lifecycle events exist, and which are missing between order and pickup?
print("Driver event types:", sorted(driver_ev.type.unique()))
print("Order -> pickup explains delay (corr):",
      round(om.loc[om.late_flag.notna(), ["first_mile_min", "delay_min"]].corr().iloc[0, 1], 2))

Driver reassignments - dispatch API: 95 | intervention log: 155 | in both: 8
Driver event types: ['assigned', 'delivered', 'gps_ping', 'picked_up']
Order -> pickup explains delay (corr): 0.83


### Challenge 6 — Answer: one-page project view

```
PROJECT KPI        Reduce Late Delivery Rate  (today 56.4% of delivered orders; 23.3% > 10 min)
     │
OUTCOME METRICS    Late Delivery Rate · Severe Late Rate (>10 min) · Cancellation rate (4.3%)
     │
WORKFLOW / DRIVER  First-mile time (order → pickup, median ~26 min, corr 0.83 with delay)
METRICS            Pickup slip vs dispatch estimated pickup · Last-mile time · ETA revisions
                   Support Contact Rate (18.3%, lagging customer signal)
     │
INTERVENTIONS      Before pickup:  DRIVER_REASSIGNMENT · PRIORITY_DISPATCH · RESTAURANT_CONTACT
                   After delivery: CUSTOMER_CREDIT  (recovery; does not move the KPI)
                   Proactive Intervention Rate (~69% before pickup)
     │
DATA SOURCES /     orders: ORDER_CREATED, PICKED_UP, DELIVERED, promised_eta
EVENTS             dispatch API: ASSIGNED, REASSIGNED, estimated_pickup, current ETA
                   driver_events: assigned, gps_ping, picked_up, delivered
                   restaurant_status (snapshot) · customer_app_actions · support_tickets
                   order_interventions · order_outcomes
```

1. **Directly controllable by operations:**
   - which interventions are made, and when (proactive intervention rate, time-to-intervene)
   - dispatch assignment speed and reassignment
   - priority dispatch
   - restaurant contact
   - acting on customer signals (the 219 frustrated journeys with no intervention)
   - First-mile time is *partly* controllable, through dispatch and restaurant-partner management.
2. **Outcomes:** Late Delivery Rate, Severe Late Rate, delay minutes and cancellation rate. Support contact rate is a customer outcome of lateness, not a lever.
3. **The missing event that limits the model most: `DRIVER_ARRIVED_AT_RESTAURANT`, together with `FOOD_READY`.**
   - The first mile explains most of the delay, but it is a black box between `ORDER_CREATED` and `PICKED_UP`.
   - We cannot tell restaurant prep delay from driver arrival delay, so we cannot pick the right intervention: restaurant contact vs reassignment.
4. **What to instrument next:**
   1. geofenced `DRIVER_ARRIVED_AT_RESTAURANT`
   2. system-generated `FOOD_READY` / `HANDED_OFF` from the restaurant (not manual)
   3. `ETA_REVISED` events that keep every value
   4. interventions with a **trigger** (risk score + timestamp) and one shared ID with dispatch
   5. mandatory `order_id` on every ticket

**Final deliverable**

| Item | Content |
|---|---|
| **Core entities** | Customer · Order · Customer Interaction (app + support) · Intervention · Outcome · (Restaurant, Driver as reference) |
| **Key events** | ORDER_CREATED → DRIVER_ASSIGNED → *(missing: DRIVER_ARRIVED, FOOD_READY)* → PICKED_UP → DELIVERED; customer: ETA_VIEWED, SUPPORT_OPENED, CANCEL_ATTEMPTED, SUPPORT_TICKET; ops: interventions |
| **Relationships** | customer 1:N order; order 1:N interaction; order 1:N intervention; order 1:1 outcome; all keyed on `order_id` |
| **Metrics** | Late Delivery Rate · Severe Late Rate · Median first-mile time · Support Contact Rate · Proactive Intervention Rate |
| **KPI linkage** | First-mile time drives lateness. Proactive interventions are the lever on first-mile time. Late delivery drives support contact. Customer credit is recovery only. |
| **Modelling limitation** | The first mile has no observed events, so delay cannot be attributed to restaurant vs driver. Interventions cannot be evaluated causally: they are targeted (selection bias), have no recorded trigger or risk score, and the reassignment log disagrees with dispatch (155 vs 95 orders, only 8 in common). |

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.